# Eksperyment 1: Porównanie klasyfikatorów i rozszerzone metryki

## Cel eksperymentu
Porównanie trzech klasyfikatorów na tym samym zbiorze danych tenisowych:
1. **Logistic Regression** — prosty model liniowy (baseline)
2. **Random Forest** — zespół drzew decyzyjnych (model bazowy z `TenisPredictionModel.ipynb`)
3. **XGBoost** — gradient boosting, często najlepszy klasyfikator tabelaryczny

## Rozszerzone metryki
Oprócz accuracy raportujemy metryki stosowane w literaturze dot. przewidywania wyników sportowych:

| Metryka | Co mierzy | Dlaczego ważna |
|---------|-----------|----------------|
| **Accuracy** | Odsetek poprawnych predykcji | Podstawowa miara |
| **Log Loss** | Jakość kalibracji prawdopodobieństw | Standardowa metryka w predykcji sportowej — karze pewne, ale błędne predykcje |
| **Brier Score** | Średni błąd kwadratowy prawdopodobieństw | Klasyczna miara w prognozowaniu (Brier 1950), im niższa tym lepiej |
| **ROC AUC** | Zdolność rozróżniania klas | Niezależna od progu decyzyjnego |
| **Cohen's Kappa** | Zgodność skorygowana o losowe trafienia | Używa się w ML, aby porównać z baseline losowym |

In [3]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, log_loss, brier_score_loss,
    roc_auc_score, cohen_kappa_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

## 1-6. Przygotowanie danych

Identyczny pipeline jak w modelu bazowym: wczytanie danych 2024, historia 2022-2023, cechy dynamiczne (expanding window), symetryzacja. Kod skondensowany w jednej komórce.

In [4]:
# --- Wczytanie danych ---
df = pd.read_csv('sample_data/2024.csv')
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d')
df = df.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)

cols_base = ['surface', 'tourney_level', 'winner_rank', 'winner_age', 'winner_ht',
             'loser_rank', 'loser_age', 'loser_ht', 'winner_name', 'loser_name']
df_base = df[cols_base].dropna().copy()
df_base['winner_rank_log'] = np.log(df_base['winner_rank'])
df_base['loser_rank_log'] = np.log(df_base['loser_rank'])

# --- Dane historyczne (cold-start) ---
history_parts = []
for filepath in ['sample_data/2022.csv', 'sample_data/2023.csv']:
    df_hist = pd.read_csv(filepath)
    df_hist['tourney_date'] = pd.to_datetime(df_hist['tourney_date'], format='%Y%m%d')
    df_hist = df_hist.sort_values(['tourney_date', 'match_num']).reset_index(drop=True)
    history_parts.append(df_hist[cols_base].dropna().copy())
df_history_base = pd.concat(history_parts, ignore_index=True)

# --- Label Encoding ---
le_surface = LabelEncoder()
le_level = LabelEncoder()
le_surface.fit(pd.concat([df_base['surface'], df_history_base['surface']]).unique())
le_level.fit(pd.concat([df_base['tourney_level'], df_history_base['tourney_level']]).unique())
df_base['surface_encoded'] = le_surface.transform(df_base['surface'])
df_base['tourney_level_encoded'] = le_level.transform(df_base['tourney_level'])

# --- Podzial chronologiczny 60/20/20 ---
train_end = int(len(df_base) * 0.60)
val_end = int(len(df_base) * 0.80)
df_train_raw = df_base.iloc[:train_end].reset_index(drop=True)
df_val_raw = df_base.iloc[train_end:val_end].reset_index(drop=True)
df_test_raw = df_base.iloc[val_end:].reset_index(drop=True)
df_train_raw['match_id'] = range(len(df_train_raw))
df_val_raw['match_id'] = range(len(df_val_raw))
df_test_raw['match_id'] = range(len(df_test_raw))

print(f"Dane 2024: {len(df_base)} meczow")
print(f"Historia (2022-2023): {len(df_history_base)} meczow")
print(f"Podzial: trening={len(df_train_raw)}, walidacja={len(df_val_raw)}, test={len(df_test_raw)}")

Dane 2024: 3005 meczow
Historia (2022-2023): 5789 meczow
Podzial: trening=1803, walidacja=601, test=601


In [5]:
# --- Cechy dynamiczne ---
def calculate_form(player_name, history):
    ph = history[(history['winner_name'] == player_name) | (history['loser_name'] == player_name)].tail(10)
    if len(ph) == 0:
        return 0.5
    return len(ph[ph['winner_name'] == player_name]) / len(ph)

def get_h2h(p1, p2, history):
    return (len(history[(history['winner_name'] == p1) & (history['loser_name'] == p2)]) -
            len(history[(history['winner_name'] == p2) & (history['loser_name'] == p1)]))

def calculate_surface_form(player_name, surface, history):
    sm = history[history['surface'] == surface]
    ps = sm[(sm['winner_name'] == player_name) | (sm['loser_name'] == player_name)].tail(10)
    if len(ps) < 3:
        return calculate_form(player_name, history)
    return len(ps[ps['winner_name'] == player_name]) / len(ps)

def add_dynamic_features(df_subset, historical_data):
    h2h_list, w_form_list, l_form_list, w_sf_list, l_sf_list = [], [], [], [], []
    full_sequence = pd.concat([historical_data, df_subset]).reset_index(drop=True)
    start_idx = len(historical_data)
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        past = full_sequence.iloc[:start_idx + i]
        h2h_list.append(get_h2h(row['winner_name'], row['loser_name'], past))
        w_form_list.append(calculate_form(row['winner_name'], past))
        l_form_list.append(calculate_form(row['loser_name'], past))
        w_sf_list.append(calculate_surface_form(row['winner_name'], row['surface'], past))
        l_sf_list.append(calculate_surface_form(row['loser_name'], row['surface'], past))
    df_subset = df_subset.copy()
    df_subset['h2h_diff'] = h2h_list
    df_subset['w_form'] = w_form_list
    df_subset['l_form'] = l_form_list
    df_subset['w_surface_form'] = w_sf_list
    df_subset['l_surface_form'] = l_sf_list
    return df_subset

df_train_raw = add_dynamic_features(df_train_raw, df_history_base)
history_val = pd.concat([df_history_base, df_train_raw[cols_base]]).reset_index(drop=True)
df_val_raw = add_dynamic_features(df_val_raw, history_val)
history_test = pd.concat([df_history_base, df_train_raw[cols_base], df_val_raw[cols_base]]).reset_index(drop=True)
df_test_raw = add_dynamic_features(df_test_raw, history_test)

print("Cechy dynamiczne obliczone.")

Cechy dynamiczne obliczone.


In [6]:
# --- Symetryzacja ---
def symmetrize_data(df_subset, shuffle=True):
    rows_p1_wins, rows_p2_wins = [], []
    for idx, row in df_subset.iterrows():
        base = {'match_id': row['match_id'], 'surface': row['surface_encoded'],
                'tourney_level': row['tourney_level_encoded']}
        row1 = {**base,
                'p1_rank_log': row['winner_rank_log'], 'p1_age': row['winner_age'], 'p1_ht': row['winner_ht'],
                'p2_rank_log': row['loser_rank_log'], 'p2_age': row['loser_age'], 'p2_ht': row['loser_ht'],
                'p1_h2h': row['h2h_diff'], 'p1_form': row['w_form'], 'p2_form': row['l_form'],
                'p1_surface_form': row['w_surface_form'], 'p2_surface_form': row['l_surface_form'],
                'rank_diff': row['winner_rank_log'] - row['loser_rank_log'],
                'age_diff': row['winner_age'] - row['loser_age'],
                'ht_diff': row['winner_ht'] - row['loser_ht'],
                'form_diff': row['w_form'] - row['l_form'],
                'y': 1, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
                'p1_name': row['winner_name'], 'p2_name': row['loser_name']}
        row2 = {**base,
                'p1_rank_log': row['loser_rank_log'], 'p1_age': row['loser_age'], 'p1_ht': row['loser_ht'],
                'p2_rank_log': row['winner_rank_log'], 'p2_age': row['winner_age'], 'p2_ht': row['winner_ht'],
                'p1_h2h': -row['h2h_diff'], 'p1_form': row['l_form'], 'p2_form': row['w_form'],
                'p1_surface_form': row['l_surface_form'], 'p2_surface_form': row['w_surface_form'],
                'rank_diff': row['loser_rank_log'] - row['winner_rank_log'],
                'age_diff': row['loser_age'] - row['winner_age'],
                'ht_diff': row['loser_ht'] - row['winner_ht'],
                'form_diff': row['l_form'] - row['w_form'],
                'y': 0, 'actual_winner': row['winner_name'], 'actual_loser': row['loser_name'],
                'p1_name': row['loser_name'], 'p2_name': row['winner_name']}
        rows_p1_wins.append(row1)
        rows_p2_wins.append(row2)
    all_rows = [r for pair in zip(rows_p1_wins, rows_p2_wins) for r in pair]
    result = pd.DataFrame(all_rows)
    if shuffle:
        result = result.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return result

train_data_ordered = symmetrize_data(df_train_raw, shuffle=False)
train_data_final = symmetrize_data(df_train_raw, shuffle=True)
val_data = symmetrize_data(df_val_raw, shuffle=True)
test_data = symmetrize_data(df_test_raw, shuffle=True)

features = ['surface', 'tourney_level', 'p1_rank_log', 'p1_age', 'p1_ht',
            'p2_rank_log', 'p2_age', 'p2_ht', 'p1_h2h', 'p1_form', 'p2_form',
            'p1_surface_form', 'p2_surface_form',
            'rank_diff', 'age_diff', 'ht_diff', 'form_diff']

X_train_cv = train_data_ordered[features]
y_train_cv = train_data_ordered['y']
X_train = train_data_final[features]
y_train = train_data_final['y']
X_val = val_data[features]
y_val = val_data['y']
X_test = test_data[features]
y_test = test_data['y']

print(f"Zbiory gotowe: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
print(f"Liczba cech: {len(features)}")

Zbiory gotowe: train=3606, val=1202, test=1202
Liczba cech: 17


## 7. Trening trzech klasyfikatorów

### Logistic Regression (baseline)
Najprostszy model liniowy — modeluje prawdopodobieństwo zwycięstwa jako funkcję liniową cech przepuszczoną przez sigmoidę. Stanowi **dolne ograniczenie** (baseline) — jeśli bardziej złożony model nie bije regresji logistycznej, to znaczy, że nie wykorzystuje nieliniowych zależności w danych.

### Random Forest
Zespół drzew decyzyjnych z baggingiem — model bazowy z głównego notebooka.

### XGBoost
Gradient boosting — buduje drzewa sekwencyjnie, każde kolejne poprawia błędy poprzedniego. Często wygrywa w konkursach ML na danych tabelarycznych.

In [7]:
# --- 1. Logistic Regression ---
# Standaryzacja cech (LR wymaga znormalizowanych danych)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)
print("Logistic Regression wytrenowana.")

Logistic Regression wytrenowana.


In [8]:
# --- 2. Random Forest (z optymalizacja hiperparametrow) ---
tscv = TimeSeriesSplit(n_splits=5)
param_dist_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 15, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True],
    'max_samples': [0.7, 0.8, 0.9, 1.0]
}

search_rf = RandomizedSearchCV(
    RandomForestClassifier(n_jobs=1, random_state=RANDOM_STATE),
    param_dist_rf, n_iter=50, cv=tscv, scoring='accuracy',
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE
)
search_rf.fit(X_train_cv, y_train_cv)

best_rf = search_rf.best_estimator_
best_rf.n_jobs = -1
best_rf.fit(X_train, y_train)
print(f"Random Forest — best CV: {search_rf.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Random Forest — best CV: 0.6426


In [9]:
# --- 3. XGBoost (z optymalizacja hiperparametrow) ---
param_dist_xgb = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 2.0, 5.0]
}

search_xgb = RandomizedSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                  random_state=RANDOM_STATE, verbosity=0),
    param_dist_xgb, n_iter=50, cv=tscv, scoring='accuracy',
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE
)
search_xgb.fit(X_train_cv, y_train_cv)

best_xgb = search_xgb.best_estimator_
best_xgb.fit(X_train, y_train)
print(f"XGBoost — best CV: {search_xgb.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
XGBoost — best CV: 0.6459


## 8. Porównanie na zbiorze testowym — rozszerzone metryki

Metryki obliczane na **symetryzowanym** zbiorze testowym (klasyfikacja binarna) oraz na **poziomie meczów** (unikalne mecze).

**Interpretacja metryk:**
- **Log Loss** — im niższa, tym lepiej skalibrowane prawdopodobieństwa. Wartości ~0.69 = losowe zgadywanie.
- **Brier Score** — MSE prawdopodobieństw. Wartość 0.25 = losowe, 0.0 = idealne.
- **ROC AUC** — 0.5 = losowe, 1.0 = idealne rozdzielenie klas.
- **Cohen's Kappa** — 0.0 = tak samo jak losowe, 1.0 = idealna zgodność.

In [10]:
def evaluate_model(model, X_test, y_test, test_data, model_name, use_scaled=False, X_test_scaled=None):
    """Oblicza pelny zestaw metryk dla modelu."""
    X = X_test_scaled if use_scaled else X_test

    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    # Metryki na symetryzowanym zbiorze
    acc = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba)
    brier = brier_score_loss(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    kappa = cohen_kappa_score(y_test, y_pred)

    # Metryka na poziomie meczow (unikalne mecze)
    td = test_data.copy()
    td['p1_win_prob'] = y_proba
    wp = td[td['y'] == 1].copy()
    match_acc = (wp['p1_win_prob'] > 0.5).mean()

    # Brier Score na poziomie meczow (perspektywa zwyciezcy)
    match_brier = ((1.0 - wp['p1_win_prob']) ** 2).mean()

    return {
        'Model': model_name,
        'Accuracy': acc,
        'Match Accuracy': match_acc,
        'Log Loss': ll,
        'Brier Score': brier,
        'Match Brier': match_brier,
        'ROC AUC': auc,
        'Cohen Kappa': kappa
    }

results = []
results.append(evaluate_model(lr, X_test, y_test, test_data, 'Logistic Regression',
                               use_scaled=True, X_test_scaled=X_test_scaled))
results.append(evaluate_model(best_rf, X_test, y_test, test_data, 'Random Forest'))
results.append(evaluate_model(best_xgb, X_test, y_test, test_data, 'XGBoost'))

df_results = pd.DataFrame(results).set_index('Model')
print(df_results.round(4).to_string())

                     Accuracy  Match Accuracy  Log Loss  Brier Score  Match Brier  ROC AUC  Cohen Kappa
Model                                                                                                  
Logistic Regression    0.6240          0.6240    0.6523       0.2305       0.2305   0.6640       0.2479
Random Forest          0.6015          0.6007    0.6601       0.2343       0.2336   0.6489       0.2030
XGBoost                0.6148          0.6040    0.6522       0.2305       0.2313   0.6583       0.2296


## 9. Szczegółowe raporty klasyfikacji

In [11]:
models = [
    ('Logistic Regression', lr, X_test_scaled),
    ('Random Forest', best_rf, X_test),
    ('XGBoost', best_xgb, X_test)
]

for name, model, X in models:
    y_pred = model.predict(X)
    print(f"\n{'='*50}")
    print(f"{name}")
    print('='*50)
    print(classification_report(y_test, y_pred, target_names=['Gracz 2 wygrywa', 'Gracz 1 wygrywa']))
    print("Macierz pomylek:")
    print(confusion_matrix(y_test, y_pred))


Logistic Regression
                 precision    recall  f1-score   support

Gracz 2 wygrywa       0.62      0.62      0.62       601
Gracz 1 wygrywa       0.62      0.62      0.62       601

       accuracy                           0.62      1202
      macro avg       0.62      0.62      0.62      1202
   weighted avg       0.62      0.62      0.62      1202

Macierz pomylek:
[[375 226]
 [226 375]]

Random Forest
                 precision    recall  f1-score   support

Gracz 2 wygrywa       0.60      0.60      0.60       601
Gracz 1 wygrywa       0.60      0.60      0.60       601

       accuracy                           0.60      1202
      macro avg       0.60      0.60      0.60      1202
   weighted avg       0.60      0.60      0.60      1202

Macierz pomylek:
[[362 239]
 [240 361]]

XGBoost
                 precision    recall  f1-score   support

Gracz 2 wygrywa       0.61      0.63      0.62       601
Gracz 1 wygrywa       0.62      0.60      0.61       601

       accur

## 10. Ważność cech — porównanie RF vs XGBoost

Random Forest używa **Mean Decrease Impurity** (Gini), XGBoost używa **gain** (suma redukcji funkcji straty przy użyciu cechy). Porównanie pokazuje, które cechy są najważniejsze z perspektywy obu algorytmów.

In [12]:
fi_rf = pd.DataFrame({'feature': features, 'RF_importance': best_rf.feature_importances_})
fi_xgb = pd.DataFrame({'feature': features, 'XGB_importance': best_xgb.feature_importances_})
fi = fi_rf.merge(fi_xgb, on='feature')
fi['avg_importance'] = (fi['RF_importance'] + fi['XGB_importance']) / 2
fi = fi.sort_values('avg_importance', ascending=False)

print(fi.round(4).to_string(index=False))

        feature  RF_importance  XGB_importance  avg_importance
      rank_diff         0.2299          0.2200          0.2250
    p1_rank_log         0.1125          0.1121          0.1123
    p2_rank_log         0.1118          0.1104          0.1111
      form_diff         0.0589          0.1146          0.0868
        p1_form         0.0334          0.0725          0.0529
        p2_form         0.0328          0.0678          0.0503
         p2_age         0.0681          0.0244          0.0463
         p1_age         0.0668          0.0251          0.0459
       age_diff         0.0649          0.0234          0.0442
p1_surface_form         0.0348          0.0454          0.0401
        ht_diff         0.0423          0.0310          0.0367
p2_surface_form         0.0345          0.0370          0.0358
          p1_ht         0.0300          0.0297          0.0299
          p2_ht         0.0296          0.0276          0.0286
  tourney_level         0.0238          0.0225         

## 11. Podsumowanie i wnioski

Na podstawie przeprowadzonego eksperymentu można wyciągnąć kilka istotnych obserwacji.

Spośród trzech testowanych modeli najlepsza trafność meczową osiągnęła **Regresja Logistyczna** — zdecydowanie najprostszy z testowanych klasyfikatorów. Na pierwszy rzut oka może to zaskakiwać, bo zarówno Random Forest, jak i XGBoost są modelami znacznie bardziej złożonymi i w wielu zastosowaniach ML radzą sobie lepiej. Jednak w przypadku predykcji wyników tenisowych mamy do czynienia z danymi z natury silnie zaszumionymi — wynik meczu w dużym stopniu zależy od czynników, których nie da się uchwycić w cechach numerycznych (dyspozycja dnia, drobne kontuzje, presja psychiczna, warunki pogodowe). W takiej sytuacji prostszy model jest mniej podatny na przeuczenie (*overfitting*) i lepiej generalizuje na nowe dane. Jest to zjawisko dobrze znane w literaturze dot. predykcji sportowej.

Warto zwrócić uwagę, że różnice między modelami nie są duże — to rząd kilku punktów procentowych, co jest typowe dla tego typu zadań. Nawet najlepsze modele opisywane w literaturze rzadko przekraczają 65-70% trafności w tenisie męskim.

Jeśli chodzi o metryki kalibracji prawdopodobieństw (Log Loss i Brier Score), również tutaj Regresja Logistyczna wypada najlepiej lub porównywalnie z XGBoost. Oznacza to, że przypisywane przez nią prawdopodobieństwa są dobrze skalibrowane — model nie jest ani nadmiernie pewny siebie, ani zbyt ostrożny w swoich predykcjach. Dla porównania, losowe zgadywanie (rzut moneta) dałoby Log Loss około 0.693 i Brier Score 0.25, więc wszystkie trzy modele wyraźnie wykraczają poza ten poziom — cechy, których użyliśmy, niosą rzeczywistą wartość predykcyjną.

Z analizy ważności cech wynika, że najsilniejszym predyktorem jest `rank_diff`, czyli różnica logarytmów rankingów ATP obu graczy. To logiczne — ranking ATP dość dobrze odzwierciedla aktualną siłę gracza. Cechy dynamiczne (forma ogólna, forma nawierzchniowa, bilans H2H) mają mniejsze, ale widoczne znaczenie wspierające — pomagają modelowi wychwycić krótkoterminowe zmiany w dyspozycji zawodników.

In [13]:
print("=" * 60)
print("PODSUMOWANIE POROWNANIA KLASYFIKATOROW")
print("=" * 60)
print()
print(df_results[['Match Accuracy', 'Log Loss', 'Brier Score', 'ROC AUC']].round(4).to_string())
print()

best_model = df_results['Match Accuracy'].idxmax()
best_acc = df_results.loc[best_model, 'Match Accuracy']
print(f"Najlepszy model wg Match Accuracy: {best_model} ({best_acc:.4f})")

best_ll = df_results['Log Loss'].idxmin()
print(f"Najlepszy model wg Log Loss:       {best_ll} ({df_results.loc[best_ll, 'Log Loss']:.4f})")

best_brier = df_results['Brier Score'].idxmin()
print(f"Najlepszy model wg Brier Score:    {best_brier} ({df_results.loc[best_brier, 'Brier Score']:.4f})")

print(f"\nBaseline losowy: Accuracy=0.5000, Log Loss=0.6931, Brier=0.2500")

PODSUMOWANIE POROWNANIA KLASYFIKATOROW

                     Match Accuracy  Log Loss  Brier Score  ROC AUC
Model                                                              
Logistic Regression          0.6240    0.6523       0.2305   0.6640
Random Forest                0.6007    0.6601       0.2343   0.6489
XGBoost                      0.6040    0.6522       0.2305   0.6583

Najlepszy model wg Match Accuracy: Logistic Regression (0.6240)
Najlepszy model wg Log Loss:       XGBoost (0.6522)
Najlepszy model wg Brier Score:    XGBoost (0.2305)

Baseline losowy: Accuracy=0.5000, Log Loss=0.6931, Brier=0.2500
